In [1]:

import sys
import itertools
from tqdm.auto import tqdm
import pathlib
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

import datasets
from contextlib import nullcontext
import torch
from torch import nn
from transformers import (
    Trainer,
    TrainingArguments,
    LlamaTokenizer,
    LlamaForSequenceClassification,
    TrainerCallback,
    default_data_collator,
)
from peft import (
    get_peft_model,
    LoraConfig,
    TaskType,
    prepare_model_for_int8_training,
)


sys.path.append("../src")
sys.path.append("../config")
from utils import number_split, create_mix


from process_HateSpeech import load_HateSpeech_dynGen, load_HateSpeech_wsf
from process_SHAC import load_process_SHAC
from process_CD import load_cd

import warnings

warnings.simplefilter("ignore")


In [2]:
df_dynGen = load_HateSpeech_dynGen()
df_wsf = load_HateSpeech_wsf()

In [9]:
df_shac = load_process_SHAC(replaceNA="all")

df_shac_uw = df_shac.query("location == 'uw'").reset_index(drop=True)
df_shac_mimic = df_shac.query("location == 'mimic'").reset_index(drop=True)


In [2]:
df_all = load_cd()
df_avh = df_all['avh']
df_r56 = df_all['r56']

In [10]:
print(df_dynGen['label_binary'].sum()/len(df_dynGen))
print(df_wsf['label_binary'].sum()/len(df_wsf))

0.5389607233132413
0.1117443707371765


In [11]:
print(df_shac_uw['Drug'].sum()/len(df_shac_uw))
print(df_shac_mimic['Drug'].sum()/len(df_shac_mimic))

0.41139240506329117
0.1976558337773042


In [4]:
print(df_avh['label_binary'].sum()/len(df_avh))
print(df_r56['label_binary'].sum()/len(df_r56))

0.2157079646017699
0.28678270329072614


In [5]:
# # ## Hate Speech
# df0 = df_dynGen
# df1 = df_wsf
# df_split_label = "label_binary"

# p_pos_train_z0_ls = np.arange(0, 1, 0.1) # probability of training set examples drawn from site/domain z0 being positive
# p_pos_train_z1_ls = np.arange(0, 1, 0.1) # probability of test set examples drawn from site/domain z1 being positive
# p_mix_z1_ls     = np.arange(0.1, 0.9, 0.1) 
# # n_test = 1000
# n_test = 200



# SHAC

# df_split_label = "Drug"

# df0 = df_shac_uw
# df1 = df_shac_mimic
# p_pos_train_z0_ls = np.arange(0, 1, 0.1)
# p_pos_train_z1_ls = np.arange(0, 1, 0.1)
# p_mix_z1_ls = np.arange(0, 1, 0.05)

# n_test = 200


# ## Hate Speech
df0 = df_avh
df1 = df_r56
df_split_label = "label_binary"

p_pos_train_z0_ls = np.arange(0, 1, 0.1) # probability of training set examples drawn from site/domain z0 being positive
p_pos_train_z1_ls = np.arange(0, 1, 0.1) # probability of test set examples drawn from site/domain z1 being positive
p_mix_z1_ls     = np.arange(0.1, 0.9, 0.1) 
n_test = 200



##### Split

train_test_ratio = 4


numvals = 1023
base = 1.1
alpha_test_ls = np.power(base, np.arange(numvals)) / np.power(base, numvals // 2)



valid_full_settings = []
for combination in itertools.product(
    p_pos_train_z0_ls, p_pos_train_z1_ls, p_mix_z1_ls, alpha_test_ls
):
    number_setting = number_split(
        p_pos_train_z0=combination[0],
        p_pos_train_z1=combination[1],
        p_mix_z1=combination[2],
        alpha_test=combination[3],
        train_test_ratio=train_test_ratio,
        n_test=n_test,
        verbose=False,
    )

    
            
    if number_setting is not None:
        if np.all([number_setting[k] >= 10 for k in list(number_setting.keys())[:-1]]):
            valid_full_settings.append(number_setting)





valid_n_full_settings = []

for c in tqdm(valid_full_settings):
        c = c.copy()
        # create train/test split according to stats
        dfs = create_mix(df0=df0, df1=df1, target=df_split_label, setting=c, sample=False, 
                         seed=222
                        )

        if dfs is None:
            continue
        
        valid_n_full_settings.append(c)

  0%|          | 0/12897 [00:00<?, ?it/s]

In [6]:
tmp = [x['mix_param_dict'] for x in valid_n_full_settings]

In [7]:
df = pd.DataFrame(tmp)

In [11]:
df.query("(alpha_train == 0.2) and (C_z == 0.5) and (p_mix_z0 == 0.5) and (0.1 <= p_pos_train_z1 <=0.2)")

,p_pos_train_z0,p_pos_train_z1,p_pos_train,p_pos_test,p_mix_z0,p_mix_z1,alpha_train,alpha_test,p_pos_test_z0,p_pos_test_z1,C_y,C_z,C_y_test
6604,0.5,0.1,0.3,0.3,0.5,0.5,0.2,0.197845,0.500900,0.099100,0.3,0.5,0.3
6605,0.5,0.1,0.3,0.3,0.5,0.5,0.2,0.217629,0.492761,0.107239,0.3,0.5,0.3
6606,0.5,0.1,0.3,0.3,0.5,0.5,0.2,0.239392,0.484108,0.115892,0.3,0.5,0.3
6607,0.5,0.1,0.3,0.3,0.5,0.5,0.2,0.263331,0.474935,0.125065,0.3,0.5,0.3
6608,0.5,0.1,0.3,0.3,0.5,0.5,0.2,0.289664,0.465237,0.134763,0.3,0.5,0.3
6609,0.5,0.1,0.3,0.3,0.5,0.5,0.2,0.318631,0.455017,0.144983,0.3,0.5,0.3
6610,0.5,0.1,0.3,0.3,0.5,0.5,0.2,0.350494,0.444282,0.155718,0.3,0.5,0.3
6611,0.5,0.1,0.3,0.3,0.5,0.5,0.2,0.385543,0.433043,0.166957,0.3,0.5,0.3
6612,0.5,0.1,0.3,0.3,0.5,0.5,0.2,0.424098,0.421319,0.178681,0.3,0.5,0.3
6613,0.5,0.1,0.3,0.3,0.5,0.5,0.2,0.466507,0.409135,0.190865,0.3,0.5,0.3


In [12]:
df.query("(alpha_train == 5) and (C_z == 0.5) and (p_mix_z0 == 0.5) and (0.1 <= p_pos_train_z0 <=0.2)")

,p_pos_train_z0,p_pos_train_z1,p_pos_train,p_pos_test,p_mix_z0,p_mix_z1,alpha_train,alpha_test,p_pos_test_z0,p_pos_test_z1,C_y,C_z,C_y_test
549,0.1,0.5,0.3,0.3,0.5,0.5,5.0,0.197845,0.500900,0.099100,0.3,0.5,0.3
550,0.1,0.5,0.3,0.3,0.5,0.5,5.0,0.217629,0.492761,0.107239,0.3,0.5,0.3
551,0.1,0.5,0.3,0.3,0.5,0.5,5.0,0.239392,0.484108,0.115892,0.3,0.5,0.3
552,0.1,0.5,0.3,0.3,0.5,0.5,5.0,0.263331,0.474935,0.125065,0.3,0.5,0.3
553,0.1,0.5,0.3,0.3,0.5,0.5,5.0,0.289664,0.465237,0.134763,0.3,0.5,0.3
554,0.1,0.5,0.3,0.3,0.5,0.5,5.0,0.318631,0.455017,0.144983,0.3,0.5,0.3
555,0.1,0.5,0.3,0.3,0.5,0.5,5.0,0.350494,0.444282,0.155718,0.3,0.5,0.3
556,0.1,0.5,0.3,0.3,0.5,0.5,5.0,0.385543,0.433043,0.166957,0.3,0.5,0.3
557,0.1,0.5,0.3,0.3,0.5,0.5,5.0,0.424098,0.421319,0.178681,0.3,0.5,0.3
558,0.1,0.5,0.3,0.3,0.5,0.5,5.0,0.466507,0.409135,0.190865,0.3,0.5,0.3


In [14]:
df.query("(alpha_train == 1) and (0.30 <= C_y <=0.31) and (C_z==0.5)")
     

,p_pos_train_z0,p_pos_train_z1,p_pos_train,p_pos_test,p_mix_z0,p_mix_z1,alpha_train,alpha_test,p_pos_test_z0,p_pos_test_z1,C_y,C_z,C_y_test
3619,0.3,0.3,0.3,0.3,0.5,0.5,1.0,0.197845,0.500900,0.099100,0.3,0.5,0.3
3620,0.3,0.3,0.3,0.3,0.5,0.5,1.0,0.217629,0.492761,0.107239,0.3,0.5,0.3
3621,0.3,0.3,0.3,0.3,0.5,0.5,1.0,0.239392,0.484108,0.115892,0.3,0.5,0.3
3622,0.3,0.3,0.3,0.3,0.5,0.5,1.0,0.263331,0.474935,0.125065,0.3,0.5,0.3
3623,0.3,0.3,0.3,0.3,0.5,0.5,1.0,0.289664,0.465237,0.134763,0.3,0.5,0.3
3624,0.3,0.3,0.3,0.3,0.5,0.5,1.0,0.318631,0.455017,0.144983,0.3,0.5,0.3
3625,0.3,0.3,0.3,0.3,0.5,0.5,1.0,0.350494,0.444282,0.155718,0.3,0.5,0.3
3626,0.3,0.3,0.3,0.3,0.5,0.5,1.0,0.385543,0.433043,0.166957,0.3,0.5,0.3
3627,0.3,0.3,0.3,0.3,0.5,0.5,1.0,0.424098,0.421319,0.178681,0.3,0.5,0.3
3628,0.3,0.3,0.3,0.3,0.5,0.5,1.0,0.466507,0.409135,0.190865,0.3,0.5,0.3


# HateSpeech - n1000

In [18]:
valid_n_full_settings[6126]

{'n_train': 4000,
 'n_test': 1000,
 'n_z0_pos_train': 600,
 'n_z0_neg_train': 1400,
 'n_z0_pos_test': 150,
 'n_z0_neg_test': 350,
 'n_z1_pos_train': 600,
 'n_z1_neg_train': 1400,
 'n_z1_pos_test': 150,
 'n_z1_neg_test': 350,
 'mix_param_dict': {'p_pos_train_z0': 0.30000000000000004,
  'p_pos_train_z1': 0.30000000000000004,
  'p_pos_train': 0.30000000000000004,
  'p_pos_test': 0.30000000000000004,
  'p_mix_z0': 0.5,
  'p_mix_z1': 0.5,
  'alpha_train': 1.0,
  'alpha_test': 1.0,
  'p_pos_test_z0': 0.30000000000000004,
  'p_pos_test_z1': 0.30000000000000004,
  'C_y': 0.30000000000000004,
  'C_z': 0.5}}

In [24]:
valid_n_full_settings[9870]

{'n_train': 4000,
 'n_test': 1000,
 'n_z0_pos_train': 1000,
 'n_z0_neg_train': 1000,
 'n_z0_pos_test': 290,
 'n_z0_neg_test': 210,
 'n_z1_pos_train': 200,
 'n_z1_neg_train': 1800,
 'n_z1_pos_test': 10,
 'n_z1_neg_test': 490,
 'mix_param_dict': {'p_pos_train_z0': 0.5,
  'p_pos_train_z1': 0.1,
  'p_pos_train': 0.3,
  'p_pos_test': 0.3,
  'p_mix_z0': 0.5,
  'p_mix_z1': 0.5,
  'alpha_train': 0.2,
  'alpha_test': 0.035584102738367304,
  'p_pos_test_z0': 0.5793831697622975,
  'p_pos_test_z1': 0.0206168302377025,
  'C_y': 0.3,
  'C_z': 0.5}}

In [13]:
valid_n_full_settings[1874]

{'n_train': 4000,
 'n_test': 1000,
 'n_z0_pos_train': 200,
 'n_z0_neg_train': 1800,
 'n_z0_pos_test': 150,
 'n_z0_neg_test': 350,
 'n_z1_pos_train': 1000,
 'n_z1_neg_train': 1000,
 'n_z1_pos_test': 150,
 'n_z1_neg_test': 350,
 'mix_param_dict': {'p_pos_train_z0': 0.1,
  'p_pos_train_z1': 0.5,
  'p_pos_train': 0.3,
  'p_pos_test': 0.3,
  'p_mix_z0': 0.5,
  'p_mix_z1': 0.5,
  'alpha_train': 5.0,
  'alpha_test': 1.0,
  'p_pos_test_z0': 0.3,
  'p_pos_test_z1': 0.3,
  'C_y': 0.3,
  'C_z': 0.5}}

# HateSpeech - n200

In [14]:
valid_n_full_settings[6621]

{'n_train': 800,
 'n_test': 200,
 'n_z0_pos_train': 200,
 'n_z0_neg_train': 200,
 'n_z0_pos_test': 30,
 'n_z0_neg_test': 70,
 'n_z1_pos_train': 40,
 'n_z1_neg_train': 360,
 'n_z1_pos_test': 30,
 'n_z1_neg_test': 70,
 'mix_param_dict': {'p_pos_train_z0': 0.5,
  'p_pos_train_z1': 0.1,
  'p_pos_train': 0.3,
  'p_pos_test': 0.3,
  'p_mix_z0': 0.5,
  'p_mix_z1': 0.5,
  'alpha_train': 0.2,
  'alpha_test': 1.0,
  'p_pos_test_z0': 0.3,
  'p_pos_test_z1': 0.3,
  'C_y': 0.3,
  'C_z': 0.5,
  'C_y_test': 0.3}}

In [36]:
valid_n_full_settings[3636]

{'n_train': 800,
 'n_test': 200,
 'n_z0_pos_train': 120,
 'n_z0_neg_train': 280,
 'n_z0_pos_test': 30,
 'n_z0_neg_test': 70,
 'n_z1_pos_train': 120,
 'n_z1_neg_train': 280,
 'n_z1_pos_test': 30,
 'n_z1_neg_test': 70,
 'mix_param_dict': {'p_pos_train_z0': 0.30000000000000004,
  'p_pos_train_z1': 0.30000000000000004,
  'p_pos_train': 0.30000000000000004,
  'p_pos_test': 0.30000000000000004,
  'p_mix_z0': 0.5,
  'p_mix_z1': 0.5,
  'alpha_train': 1.0,
  'alpha_test': 1.0,
  'p_pos_test_z0': 0.30000000000000004,
  'p_pos_test_z1': 0.30000000000000004,
  'C_y': 0.30000000000000004,
  'C_z': 0.5,
  'C_y_test': 0.30000000000000004}}

In [8]:
valid_n_full_settings[566]

{'n_train': 800,
 'n_test': 200,
 'n_z0_pos_train': 40,
 'n_z0_neg_train': 360,
 'n_z0_pos_test': 30,
 'n_z0_neg_test': 70,
 'n_z1_pos_train': 200,
 'n_z1_neg_train': 200,
 'n_z1_pos_test': 30,
 'n_z1_neg_test': 70,
 'mix_param_dict': {'p_pos_train_z0': 0.1,
  'p_pos_train_z1': 0.5,
  'p_pos_train': 0.3,
  'p_pos_test': 0.3,
  'p_mix_z0': 0.5,
  'p_mix_z1': 0.5,
  'alpha_train': 5.0,
  'alpha_test': 1.0,
  'p_pos_test_z0': 0.3,
  'p_pos_test_z1': 0.3,
  'C_y': 0.3,
  'C_z': 0.5,
  'C_y_test': 0.3}}

# SHAC

In [26]:
valid_n_full_settings[6114]

{'n_train': 800,
 'n_test': 200,
 'n_z0_pos_train': 120,
 'n_z0_neg_train': 280,
 'n_z0_pos_test': 30,
 'n_z0_neg_test': 70,
 'n_z1_pos_train': 120,
 'n_z1_neg_train': 280,
 'n_z1_pos_test': 30,
 'n_z1_neg_test': 70,
 'mix_param_dict': {'p_pos_train_z0': 0.30000000000000004,
  'p_pos_train_z1': 0.30000000000000004,
  'p_pos_train': 0.30000000000000004,
  'p_pos_test': 0.30000000000000004,
  'p_mix_z0': 0.5,
  'p_mix_z1': 0.5,
  'alpha_train': 1.0,
  'alpha_test': 1.0,
  'p_pos_test_z0': 0.30000000000000004,
  'p_pos_test_z1': 0.30000000000000004,
  'C_y': 0.30000000000000004,
  'C_z': 0.5}}

In [32]:
valid_n_full_settings[11063]

{'n_train': 800,
 'n_test': 200,
 'n_z0_pos_train': 200,
 'n_z0_neg_train': 200,
 'n_z0_pos_test': 30,
 'n_z0_neg_test': 70,
 'n_z1_pos_train': 40,
 'n_z1_neg_train': 360,
 'n_z1_pos_test': 30,
 'n_z1_neg_test': 70,
 'mix_param_dict': {'p_pos_train_z0': 0.5,
  'p_pos_train_z1': 0.1,
  'p_pos_train': 0.3,
  'p_pos_test': 0.3,
  'p_mix_z0': 0.5,
  'p_mix_z1': 0.5,
  'alpha_train': 0.2,
  'alpha_test': 1.0,
  'p_pos_test_z0': 0.3,
  'p_pos_test_z1': 0.3,
  'C_y': 0.3,
  'C_z': 0.5}}

In [34]:
valid_n_full_settings[1152]

{'n_train': 800,
 'n_test': 200,
 'n_z0_pos_train': 40,
 'n_z0_neg_train': 360,
 'n_z0_pos_test': 30,
 'n_z0_neg_test': 70,
 'n_z1_pos_train': 200,
 'n_z1_neg_train': 200,
 'n_z1_pos_test': 30,
 'n_z1_neg_test': 70,
 'mix_param_dict': {'p_pos_train_z0': 0.1,
  'p_pos_train_z1': 0.5,
  'p_pos_train': 0.3,
  'p_pos_test': 0.3,
  'p_mix_z0': 0.5,
  'p_mix_z1': 0.5,
  'alpha_train': 5.0,
  'alpha_test': 1.0,
  'p_pos_test_z0': 0.3,
  'p_pos_test_z1': 0.3,
  'C_y': 0.3,
  'C_z': 0.5}}

In [13]:
valid_n_full_settings[2800]

{'n_train': 800,
 'n_test': 200,
 'n_z0_pos_train': 80,
 'n_z0_neg_train': 320,
 'n_z0_pos_test': 20,
 'n_z0_neg_test': 80,
 'n_z1_pos_train': 80,
 'n_z1_neg_train': 320,
 'n_z1_pos_test': 20,
 'n_z1_neg_test': 80,
 'mix_param_dict': {'p_pos_train_z0': 0.2,
  'p_pos_train_z1': 0.2,
  'p_pos_train': 0.2,
  'p_pos_test': 0.2,
  'p_mix_z0': 0.5,
  'p_mix_z1': 0.5,
  'alpha_train': 1.0,
  'alpha_test': 1.0,
  'p_pos_test_z0': 0.2,
  'p_pos_test_z1': 0.2,
  'C_y': 0.2,
  'C_z': 0.5}}

In [12]:
valid_n_full_settings[1355]

{'n_train': 800,
 'n_test': 200,
 'n_z0_pos_train': 64,
 'n_z0_neg_train': 576,
 'n_z0_pos_test': 23,
 'n_z0_neg_test': 137,
 'n_z1_pos_train': 96,
 'n_z1_neg_train': 64,
 'n_z1_pos_test': 17,
 'n_z1_neg_test': 23,
 'mix_param_dict': {'p_pos_train_z0': 0.1,
  'p_pos_train_z1': 0.6000000000000001,
  'p_pos_train': 0.20000000000000004,
  'p_pos_test': 0.20000000000000004,
  'p_mix_z0': 0.8,
  'p_mix_z1': 0.2,
  'alpha_train': 6.000000000000001,
  'alpha_test': 2.8531167061100025,
  'p_pos_test_z0': 0.145919009245594,
  'p_pos_test_z1': 0.41632396301762414,
  'C_y': 0.20000000000000004,
  'C_z': 0.2}}

In [11]:
valid_n_full_settings[13755]

{'n_train': 800,
 'n_test': 200,
 'n_z0_pos_train': 240,
 'n_z0_neg_train': 160,
 'n_z0_pos_test': 35,
 'n_z0_neg_test': 65,
 'n_z1_pos_train': 40,
 'n_z1_neg_train': 360,
 'n_z1_pos_test': 35,
 'n_z1_neg_test': 65,
 'mix_param_dict': {'p_pos_train_z0': 0.6000000000000001,
  'p_pos_train_z1': 0.1,
  'p_pos_train': 0.35000000000000003,
  'p_pos_test': 0.35000000000000003,
  'p_mix_z0': 0.5,
  'p_mix_z1': 0.5,
  'alpha_train': 0.16666666666666666,
  'alpha_test': 1.0,
  'p_pos_test_z0': 0.35000000000000003,
  'p_pos_test_z1': 0.35000000000000003,
  'C_y': 0.35000000000000003,
  'C_z': 0.5}}

# CD

In [9]:
valid_n_full_settings[6621]

{'n_train': 800,
 'n_test': 200,
 'n_z0_pos_train': 200,
 'n_z0_neg_train': 200,
 'n_z0_pos_test': 30,
 'n_z0_neg_test': 70,
 'n_z1_pos_train': 40,
 'n_z1_neg_train': 360,
 'n_z1_pos_test': 30,
 'n_z1_neg_test': 70,
 'mix_param_dict': {'p_pos_train_z0': 0.5,
  'p_pos_train_z1': 0.1,
  'p_pos_train': 0.3,
  'p_pos_test': 0.3,
  'p_mix_z0': 0.5,
  'p_mix_z1': 0.5,
  'alpha_train': 0.2,
  'alpha_test': 1.0,
  'p_pos_test_z0': 0.3,
  'p_pos_test_z1': 0.3,
  'C_y': 0.3,
  'C_z': 0.5,
  'C_y_test': 0.3}}

In [15]:
valid_n_full_settings[3636]

{'n_train': 800,
 'n_test': 200,
 'n_z0_pos_train': 120,
 'n_z0_neg_train': 280,
 'n_z0_pos_test': 30,
 'n_z0_neg_test': 70,
 'n_z1_pos_train': 120,
 'n_z1_neg_train': 280,
 'n_z1_pos_test': 30,
 'n_z1_neg_test': 70,
 'mix_param_dict': {'p_pos_train_z0': 0.30000000000000004,
  'p_pos_train_z1': 0.30000000000000004,
  'p_pos_train': 0.30000000000000004,
  'p_pos_test': 0.30000000000000004,
  'p_mix_z0': 0.5,
  'p_mix_z1': 0.5,
  'alpha_train': 1.0,
  'alpha_test': 1.0,
  'p_pos_test_z0': 0.30000000000000004,
  'p_pos_test_z1': 0.30000000000000004,
  'C_y': 0.30000000000000004,
  'C_z': 0.5,
  'C_y_test': 0.30000000000000004}}

In [13]:
valid_n_full_settings[566]

{'n_train': 800,
 'n_test': 200,
 'n_z0_pos_train': 40,
 'n_z0_neg_train': 360,
 'n_z0_pos_test': 30,
 'n_z0_neg_test': 70,
 'n_z1_pos_train': 200,
 'n_z1_neg_train': 200,
 'n_z1_pos_test': 30,
 'n_z1_neg_test': 70,
 'mix_param_dict': {'p_pos_train_z0': 0.1,
  'p_pos_train_z1': 0.5,
  'p_pos_train': 0.3,
  'p_pos_test': 0.3,
  'p_mix_z0': 0.5,
  'p_mix_z1': 0.5,
  'alpha_train': 5.0,
  'alpha_test': 1.0,
  'p_pos_test_z0': 0.3,
  'p_pos_test_z1': 0.3,
  'C_y': 0.3,
  'C_z': 0.5,
  'C_y_test': 0.3}}